# Notebook 5 — National RSF: train + evaluate

Trains the Random Survival Forest on the Notebook 4 matrix (`us_rsf_data_clean.csv`)
and evaluates it, porting the validated Massachusetts pipeline (`../ma_bridges/RSF.ipynb`).
The Notebook 4 population excludes culverts and other structures whose deck/super/sub
condition codes are all "N" (the event is undefined for them — see Notebook 4's
diagnostic cell), so the matrix is ~1M non-culvert bridges.

**Ported from MA unchanged:** the survival target (`Surv(event, time)`; "poor" = any of
deck / superstructure / substructure condition ≤ 4), the 80/20 random split
(`random_state=42`), `max_features="sqrt"`, the median-crossing predicted failure age
(first age where S(t) < 0.5), 5/10/20-year failure probabilities clipped to the
survival-function domain, and joblib model persistence.

**Intentional national deviations:**
1. **No maintenance features** — MA-DOT `maintenance.csv` has no 50-state equivalent;
   Notebook 4 already substitutes the NBI reconstruction proxies (`ever_reconstructed`,
   `years_since_reconstruction`, both measured at panel entry).
2. **`bridge_age` dropped from X** — in the Notebook 4 file it *equals* `time` by
   construction, so keeping it would leak the target (Notebook 4 prints this warning
   when it saves the CSV).
3. **Scale (~1M bridges vs MA's ~16k):** per-tree subsampling (`max_samples=0.10`)
   with 200 trees keeps the total workload near MA's 1000 full-bootstrap trees; larger
   leaves (`min_samples_leaf=50`) keep the in-RAM forest ~2 GB instead of ~20 GB;
   predictions run **chunked + vectorized** instead of MA's per-bridge Python loop;
   features load as float32. (How much C-index this coarsening costs is bounded by
   the cost-matched Arm D in `ablation_state_dummies.ipynb`.)
4. **Evaluation expanded** beyond MA's single C-index — integrated Brier score,
   time-dependent AUC, per-state C-index, optional permutation importance, plus a
   test-risk-score artifact consumed by `bootstrap_model_comparison.ipynb`.

**Run modes:** cells execute top to bottom. `SMOKE_TEST = True` trains on a
20k-bridge subsample with 25 trees and writes `*_smoke`-suffixed outputs — a cheap
per-tree cost measurement that never overwrites real artifacts. `SMOKE_TEST = False`
is the full configuration.

In [1]:
# ── Configuration ─────────────────────────────────────────────────────────────
import json
import time
import numpy as np
import pandas as pd
import joblib
from pathlib import Path

from sklearn.model_selection import train_test_split
from sksurv.ensemble import RandomSurvivalForest
from sksurv.util import Surv
from sksurv.metrics import (concordance_index_censored, integrated_brier_score, cumulative_dynamic_auc)

# Inputs (written by Notebook 4, build_national_rsf_dataset.ipynb)
DATA_CLEAN = Path("us_rsf_data_clean.csv")   # encoded model matrix
DATA_A     = Path("us_rsf_data_a.csv")       # raw per-bridge table (county, lat/long) for mapping

KEYS = ["STATE_FIPS", "STRUCTURE_NUMBER_008"]   # structure # is unique only WITHIN a state
CURRENT_YEAR = 2025          # last survey year in the data; MA used 2025 too
PRED_CHUNK   = 50_000        # bridges per predict_survival_function block

# Run-mode knobs
SMOKE_TEST     = False   # True -> small run on the Notebook 4 _smoke matrices (see below)
TRAIN_SAMPLE   = None    # e.g. 200_000 -> stratified (state, event) subsample of the training rows
RUN_IMPORTANCE = True    # permutation importance: ~15-60 min at full scale
RUN_TUNING     = False   # RandomizedSearchCV near the end; multiple hours even scaled down
DROP_STATE_DUMMIES = False   # exclude STATE_* one-hots from X (decision basis: ablation_state_dummies.ipynb)

# Scaled from MA (1000 trees, leaf=5 on ~16k bridges): max_samples=0.10 matches the
# per-tree workload, leaf=50 holds the forest near 2 GB (leaf=5 would be ~20 GB here).
# low_memory must stay False or predict_survival_function is disabled. Arm D in
# ablation_state_dummies.ipynb bounds the C-index cost of this coarsening.
RSF_PARAMS = dict(
    n_estimators=200,
    max_samples=0.10,
    min_samples_split=100,
    min_samples_leaf=50,
    max_features="sqrt",
    low_memory=False,
    n_jobs=-1,
    random_state=42,
)

if SMOKE_TEST:
    RSF_PARAMS.update(n_estimators=25, max_samples=0.5)
    # Prefer the Notebook 4 smoke matrices so the whole smoke chain runs on the same files.
    if Path("us_rsf_data_clean_smoke.csv").exists():
        DATA_CLEAN = Path("us_rsf_data_clean_smoke.csv")
        DATA_A     = Path("us_rsf_data_a_smoke.csv")
        print(f"SMOKE_TEST: reading Notebook 4 smoke matrices ({DATA_CLEAN})")

def out_path(name):
    """Output path; smoke runs get a _smoke suffix so real artifacts are never clobbered."""
    p = Path(name)
    return p.with_stem(p.stem + "_smoke") if SMOKE_TEST else p

OUT_MODEL      = out_path("us_rsf_model.pkl")
OUT_METRICS    = out_path("us_rsf_metrics.json")
OUT_STATE_CIDX = out_path("us_rsf_cindex_by_state.csv")
OUT_IMPORTANCE = out_path("us_rsf_feature_importance.csv")
OUT_YEARS      = out_path("us_results_years_until_failure.csv")
OUT_PROBS      = out_path("us_results_prob_of_failure.csv")
OUT_MAP        = out_path("us_bridge_risk_map.json")

if not DATA_CLEAN.exists():
    raise FileNotFoundError(
        f"{DATA_CLEAN} not found — it is produced by build_national_rsf_dataset.ipynb "
        "(Notebook 4), which itself requires the Notebook 2/3 climate and coastal outputs.")

print(f"config OK — smoke={SMOKE_TEST}, params={RSF_PARAMS}")

config OK — smoke=False, params={'n_estimators': 200, 'max_samples': 0.1, 'min_samples_split': 100, 'min_samples_leaf': 50, 'max_features': 'sqrt', 'low_memory': False, 'n_jobs': -1, 'random_state': 42}


In [2]:
# ── Typed load: IDs as str, everything else float32 (halves the ~1M x ~250 frame) ──
cols = pd.read_csv(DATA_CLEAN, nrows=0).columns
dtypes = {c: "float32" for c in cols}
dtypes.update({k: str for k in KEYS})
dtypes["event"] = "int8"
df = pd.read_csv(DATA_CLEAN, dtype=dtypes)
for k in KEYS:
    df[k] = df[k].str.strip()

# Subsample only under smoke against the FULL matrix; the N4 smoke matrices are used
# whole to keep the split aligned with the leakage/bootstrap notebooks.
if SMOKE_TEST and "smoke" not in DATA_CLEAN.stem and len(df) > 20_000:
    df = df.sample(n=20_000, random_state=42).reset_index(drop=True)
    print(f"SMOKE_TEST: subsampled to {len(df):,} bridges")

assert (df["time"] > 0).all(), "non-positive survival times — Notebook 4 should have dropped these"

y   = Surv.from_arrays(event=df["event"].astype(bool).to_numpy(),
                       time=df["time"].astype("float64").to_numpy())
ids = df[KEYS].copy()

# bridge_age == time here — must drop or it leaks the target.
X = df.drop(columns=["event", "time", "bridge_age"] + KEYS)

if DROP_STATE_DUMMIES:
    state_cols = [c for c in X.columns if c.startswith("STATE_")]
    X = X.drop(columns=state_cols)
    print(f"DROP_STATE_DUMMIES: removed {len(state_cols)} STATE_* columns "
          "(geography stays encoded in the climate/coastal features)")

leak = [c for c in X.columns if "_COND_" in c or c == "OPEN_CLOSED_POSTED_041"]
assert not leak, f"leakage columns present: {leak}"
assert "STRUCTURE_TYPE_043B_19" not in X.columns, \
    "culvert dummy present — the matrix predates the Notebook 4 culvert exclusion; rebuild it"
assert X.select_dtypes(exclude="number").shape[1] == 0, "non-numeric feature columns in X"

del df
print(f"X: {X.shape[0]:,} bridges x {X.shape[1]} features   "
      f"event rate: {y['event'].mean()*100:.1f}%   NaN share: {X.isna().mean().mean()*100:.1f}%")

X: 973,905 bridges x 255 features   event rate: 29.3%   NaN share: 1.4%


In [3]:
# ── 80/20 split (plain random, as in MA) + optional stratified training subsample ──
X_train, X_test, y_train, y_test, ids_train, ids_test = train_test_split(
    X, y, ids, test_size=0.2, random_state=42)

if TRAIN_SAMPLE and TRAIN_SAMPLE < len(X_train):
    strat = pd.DataFrame({"state": ids_train["STATE_FIPS"].to_numpy(),
                          "event": y_train["event"]})            # RangeIndex -> positional
    pos = (strat.groupby(["state", "event"], group_keys=False)
                .sample(frac=TRAIN_SAMPLE / len(strat), random_state=42)).index.to_numpy()
    X_train, y_train, ids_train = X_train.iloc[pos], y_train[pos], ids_train.iloc[pos]
    print(f"TRAIN_SAMPLE: training on {len(X_train):,} rows (stratified by state x event)")

print(f"train: {len(X_train):,}   test: {len(X_test):,}")

train: 779,124   test: 194,781


In [4]:
# ── Fit ───────────────────────────────────────────────────────────────────────
t0 = time.time()
rsf = RandomSurvivalForest(**RSF_PARAMS)
rsf.fit(X_train, y_train)
fit_seconds = time.time() - t0
print(f"fitted {RSF_PARAMS['n_estimators']} trees in {fit_seconds/60:.1f} min   "
      f"unique event times: {len(rsf.unique_times_)}")

# Survival matrices are (n_bridges x len(unique_times_)); integer-year ages keep this
# ~300. A trip here means `time` went fractional and memory blows up.
assert len(rsf.unique_times_) < 1500, "unexpectedly many unique event times"

fitted 200 trees in 39.1 min   unique event times: 219


In [5]:
# ── Persist the model ─────────────────────────────────────────────────────────
joblib.dump(rsf, OUT_MODEL, compress=3)
print(f"saved {OUT_MODEL} ({OUT_MODEL.stat().st_size/1e6:.0f} MB)")

saved us_rsf_model.pkl (181 MB)


## Evaluation

MA's `RSF.ipynb` reported a single metric (test C-index). At national scale the cheap,
high-value survival metrics are added. Costs below are for the full ~1M-bridge run:

| Metric | Cost | Notes |
|---|---|---|
| C-index (test) | minutes | MA-parity headline metric; risk scores cached and reused below |
| Integrated Brier score | ~5–15 min | calibration + discrimination combined; 0.25 ≈ random |
| Time-dependent AUC (25/50/75 yr) | ≈ free after IBS | auto-skipped on a known sksurv censoring edge case |
| C-index per state | seconds | checks that the single national model generalizes to every state |
| Permutation importance | ~15–60 min | optional (`RUN_IMPORTANCE`); one-hot blocks dilute per-dummy scores |

In [6]:
# ── Test C-index (the MA-parity metric) ───────────────────────────────────────
# Cached ensemble risk score, reused by the AUC and per-state cells.
risk_test = np.concatenate([rsf.predict(X_test.iloc[i:i+PRED_CHUNK])
                            for i in range(0, len(X_test), PRED_CHUNK)])
c_index = concordance_index_censored(y_test["event"], y_test["time"], risk_test)[0]
print(f"C-index (test): {c_index:.3f}")

C-index (test): 0.753


In [7]:
# ── Test-risk handoff -> us_rsf_test_risk.csv ─────────────────────────────────
# Per-bridge held-out risk, consumed by bootstrap_model_comparison.ipynb.
OUT_TEST_RISK = out_path("us_rsf_test_risk.csv")
pd.DataFrame({
    "STATE_FIPS": ids_test["STATE_FIPS"].to_numpy(),
    "STRUCTURE_NUMBER_008": ids_test["STRUCTURE_NUMBER_008"].to_numpy(),
    "event": y_test["event"].astype(int),
    "time": y_test["time"],
    "rsf_risk": risk_test,
}).to_csv(OUT_TEST_RISK, index=False)
print(f"saved {OUT_TEST_RISK} ({len(risk_test):,} test bridges)")

saved us_rsf_test_risk.csv (194,781 test bridges)


In [8]:
# ── Integrated Brier score + time-dependent AUC (chunked survival matrices) ───
times = rsf.unique_times_
S_test = np.vstack([rsf.predict_survival_function(X_test.iloc[i:i+PRED_CHUNK], return_array=True)
                    for i in range(0, len(X_test), PRED_CHUNK)])

# Grid: inner percentiles of test follow-up, snapped to the model's time grid and kept
# inside both samples' observed ranges (sksurv requires it).
tmin = max(y_train["time"].min(), y_test["time"].min())
tmax = min(y_train["time"].max(), y_test["time"].max())
grid = np.percentile(y_test["time"], np.arange(10, 91, 10))
grid = np.unique(times[np.clip(np.searchsorted(times, grid, side="right") - 1, 0, len(times) - 1)])
grid = grid[(grid > tmin) & (grid < tmax)]
gi = np.searchsorted(times, grid, side="right") - 1

ibs = integrated_brier_score(y_train, y_test, S_test[:, gi], grid)
print(f"Integrated Brier score over {grid.min():.0f}-{grid.max():.0f} yr: {ibs:.4f}   (0.25 = random)")

auc_dict, mean_auc = None, None
try:
    horizons = np.array([25.0, 50.0, 75.0])
    horizons = horizons[(horizons > tmin) & (horizons < tmax)]
    hi = np.searchsorted(times, horizons, side="right") - 1
    with np.errstate(invalid="ignore"):   # a horizon with no events yields NaN, not an error
        auc, mean_auc = cumulative_dynamic_auc(y_train, y_test, 1 - S_test[:, hi], horizons)
    auc_dict = {f"{int(h)}yr": (None if np.isnan(a) else round(float(a), 4))
                for h, a in zip(horizons, auc)}
    mean_auc = None if np.isnan(mean_auc) else float(mean_auc)   # keep the JSON strictly valid
    print("time-dependent AUC:", auc_dict, "  mean:", mean_auc)
except ValueError as e:
    print("AUC skipped (known sksurv censoring-distribution edge case):", e)

del S_test

Integrated Brier score over 13-78 yr: 0.0761   (0.25 = random)


time-dependent AUC: {'25yr': 0.9451, '50yr': 0.9535, '75yr': 0.9281}   mean: 0.9383276708876059


In [9]:
# ── C-index per state (generalization check; reuses the cached risk scores) ───
rows = []
state_arr = ids_test["STATE_FIPS"].to_numpy()
for st in np.unique(state_arr):
    m = state_arr == st
    n, n_ev = int(m.sum()), int(y_test["event"][m].sum())
    if n < 200 or n_ev < 30:      # too few comparable pairs for a stable estimate
        continue
    try:
        ci = concordance_index_censored(y_test["event"][m], y_test["time"][m], risk_test[m])[0]
        rows.append({"STATE_FIPS": st, "n_test": n, "n_events": n_ev, "c_index": round(ci, 4)})
    except Exception as e:
        print(f"  {st}: skipped ({e})")

state_cidx = (pd.DataFrame(rows, columns=["STATE_FIPS", "n_test", "n_events", "c_index"])
                .sort_values("c_index").reset_index(drop=True))
state_cidx.to_csv(OUT_STATE_CIDX, index=False)
print(f"saved {OUT_STATE_CIDX} ({len(state_cidx)} states)   "
      f"range: {state_cidx['c_index'].min():.3f}-{state_cidx['c_index'].max():.3f}")
state_cidx

saved us_rsf_cindex_by_state.csv (50 states)   range: 0.564-0.887


,STATE_FIPS,n_test,n_events,c_index
0,46,1259,710,0.5640
1,29,12289,3049,0.6102
2,40,9787,4071,0.6373
3,56,723,260,0.6395
4,10,220,63,0.6522
5,25,3836,813,0.6535
6,17,15019,3305,0.6740
7,19,10320,3366,0.6778
8,27,2934,1035,0.6782
9,42,11032,4236,0.6867


In [10]:
# ── Permutation importance (guarded by RUN_IMPORTANCE) ────────────────────────
# (n_features x n_repeats) scorings on a 10k subsample (~15-60 min). n_jobs=1 on
# purpose — parallel workers would each pickle the multi-GB forest.
if RUN_IMPORTANCE:
    from sklearn.inspection import permutation_importance
    sub = np.random.default_rng(42).choice(len(X_test), size=min(10_000, len(X_test)),
                                           replace=False)
    t0 = time.time()
    pi = permutation_importance(rsf, X_test.iloc[sub], y_test[sub],
                                n_repeats=3, random_state=42, n_jobs=1)
    importance = (pd.DataFrame({"feature": X_test.columns,
                                "importance_mean": pi.importances_mean,
                                "importance_std": pi.importances_std})
                    .sort_values("importance_mean", ascending=False).reset_index(drop=True))
    importance.to_csv(OUT_IMPORTANCE, index=False)
    print(f"saved {OUT_IMPORTANCE} ({(time.time()-t0)/60:.1f} min)\n")
    print(importance.head(20).to_string(index=False))
else:
    print("skipped (RUN_IMPORTANCE = False)")

saved us_rsf_feature_importance.csv (11.8 min)

                   feature  importance_mean  importance_std
      BASE_HWY_NETWORK_012         0.027955        0.001134
        HIGHWAY_SYSTEM_104         0.011444        0.000425
               DECK_TYPE_1         0.004661        0.000389
        SCOUR_CRITICAL_113         0.003886        0.000224
           LAT_UND_MT_055B         0.003809        0.000332
      OPERATING_RATING_064         0.003800        0.000664
            SURFACE_TYPE_1         0.003603        0.000295
      INVENTORY_RATING_066         0.003601        0.000800
                   mean_vp         0.003083        0.000137
years_since_reconstruction         0.002953        0.000163
     STRUCTURE_TYPE_043B_2         0.002851        0.000144
            YEAR_BUILT_027         0.002739        0.000170
        ever_reconstructed         0.002508        0.000032
                 snow_days         0.002468        0.000091
            freezing_index         0.002461        0

In [11]:
# ── Metrics summary -> us_rsf_metrics.json ────────────────────────────────────
metrics = {
    "generated_utc": time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime()),
    "smoke_test": SMOKE_TEST,
    "n_bridges": int(len(X)),
    "n_train": int(len(X_train)),
    "n_test": int(len(X_test)),
    "n_features": int(X.shape[1]),
    "event_rate": round(float(y["event"].mean()), 4),
    "rsf_params": {k: (v if isinstance(v, (int, float, str, bool, type(None))) else str(v))
                   for k, v in RSF_PARAMS.items()},
    "fit_seconds": round(fit_seconds, 1),
    "c_index_test": round(float(c_index), 4),
    "integrated_brier_score": round(float(ibs), 4),
    "ibs_time_grid": [float(t) for t in grid],
    "auc_by_horizon": auc_dict,
    "auc_mean": None if mean_auc is None else round(float(mean_auc), 4),
}
OUT_METRICS.write_text(json.dumps(metrics, indent=2))
print(f"saved {OUT_METRICS}\n")
print(json.dumps(metrics, indent=2))

saved us_rsf_metrics.json

{
  "generated_utc": "2026-07-20T05:40:43Z",
  "smoke_test": false,
  "n_bridges": 973905,
  "n_train": 779124,
  "n_test": 194781,
  "n_features": 255,
  "event_rate": 0.2927,
  "rsf_params": {
    "n_estimators": 200,
    "max_samples": 0.1,
    "min_samples_split": 100,
    "min_samples_leaf": 50,
    "max_features": "sqrt",
    "low_memory": false,
    "n_jobs": -1,
    "random_state": 42
  },
  "fit_seconds": 2345.1,
  "c_index_test": 0.7526,
  "integrated_brier_score": 0.0761,
  "ibs_time_grid": [
    13.0,
    22.0,
    30.0,
    36.0,
    42.0,
    50.0,
    57.0,
    64.0,
    78.0
  ],
  "auc_by_horizon": {
    "25yr": 0.9451,
    "50yr": 0.9535,
    "75yr": 0.9281
  },
  "auc_mean": 0.9383
}


## Score every bridge

Same definitions as MA:

- **Predicted failure age** — the first age where the bridge's survival curve drops
  below 0.5. Bridges whose curve never crosses 0.5 get no predicted year and are
  labeled *Low Risk*.
- **5/10/20-year failure probabilities** — `1 − S(current_age + h)` with
  `current_age = CURRENT_YEAR − YEAR_BUILT_027`, queried inside the survival-function
  domain (clipped, exactly as MA did).

Unlike MA (a per-bridge Python loop over ~20k bridges), this scores in vectorized
`PRED_CHUNK`-sized blocks — roughly 10–30 minutes at ~1M bridges.

In [12]:
# ── Chunked all-bridge predictions -> the two RESULTS CSVs ────────────────────
times = rsf.unique_times_
year_built  = X["YEAR_BUILT_027"].to_numpy(dtype="float64")
current_age = CURRENT_YEAR - year_built

parts, t0 = [], time.time()
for i in range(0, len(X), PRED_CHUNK):
    S = rsf.predict_survival_function(X.iloc[i:i+PRED_CHUNK], return_array=True)  # (m, T)

    below = S < 0.5                                  # first age where S(t) < 0.5, as in MA
    pred_age = np.where(below.any(axis=1), times[below.argmax(axis=1)], np.nan)

    ca = current_age[i:i+PRED_CHUNK]
    chunk_cols = {"predicted_age_at_poor": pred_age}
    for h in (5, 10, 20):
        target = np.clip(ca + h, times[0], times[-1])         # clip to the SF domain, as in MA
        j = np.clip(np.searchsorted(times, target, side="right") - 1, 0, len(times) - 1)
        p = 1.0 - S[np.arange(len(S)), j]
        chunk_cols[f"prob_fail_{h}yr"] = np.where(np.isnan(ca), np.nan, p).round(3)
    parts.append(pd.DataFrame(chunk_cols))
    print(f"  scored {min(i+PRED_CHUNK, len(X)):,}/{len(X):,}  ({time.time()-t0:.0f}s)", end="\r")

res = pd.concat(parts, ignore_index=True)
del parts
res.insert(0, "STRUCTURE_NUMBER_008", ids["STRUCTURE_NUMBER_008"].to_numpy())
res.insert(0, "STATE_FIPS", ids["STATE_FIPS"].to_numpy())
res["YEAR_BUILT_027"] = year_built
res["current_age"]    = current_age
res["predicted_year_goes_poor"] = (res["YEAR_BUILT_027"] + res["predicted_age_at_poor"]).round()
res["years_remaining"]          = res["predicted_year_goes_poor"] - CURRENT_YEAR
res["risk_category"] = np.where(res["predicted_age_at_poor"].notna(), "High Risk", "Low Risk")

# County / place / raw coordinates come from the raw per-bridge table (kept out of X).
ctx = pd.read_csv(DATA_A,
                  usecols=KEYS + ["COUNTY_CODE_003", "PLACE_CODE_004", "LAT_016", "LONG_017"],
                  dtype={c: str for c in KEYS + ["COUNTY_CODE_003", "PLACE_CODE_004",
                                                 "LAT_016", "LONG_017"]})
for k in KEYS:
    ctx[k] = ctx[k].str.strip()
ctx = ctx.drop_duplicates(KEYS, keep="first")
res = res.merge(ctx, on=KEYS, how="left", validate="m:1")

# MA-parity schema (RESULTS_years_until_failure.csv) + STATE_FIPS, since structure
# numbers are only unique within a state.
res[["STATE_FIPS", "STRUCTURE_NUMBER_008", "YEAR_BUILT_027", "COUNTY_CODE_003",
     "PLACE_CODE_004", "LAT_016", "LONG_017", "predicted_age_at_poor",
     "predicted_year_goes_poor", "years_remaining", "risk_category"]].to_csv(OUT_YEARS, index=False)
res.to_csv(OUT_PROBS, index=False)
print(f"\nsaved {OUT_YEARS} and {OUT_PROBS} ({len(res):,} bridges)")
print(res[["prob_fail_5yr", "prob_fail_10yr", "prob_fail_20yr"]].describe().round(3))


saved us_results_years_until_failure.csv and us_results_prob_of_failure.csv (973,905 bridges)
       prob_fail_5yr  prob_fail_10yr  prob_fail_20yr
count     973905.000      973905.000      973905.000
mean           0.418           0.448           0.497
std            0.319           0.320           0.314
min            0.000           0.000           0.000
25%            0.099           0.134           0.209
50%            0.391           0.438           0.508
75%            0.704           0.739           0.784
max            0.997           0.997           0.997


In [13]:
# ── Map JSON (plotly input) -> us_bridge_risk_map.json ────────────────────────
# Raw NBI LAT_016/LONG_017 are DDMMSS.SS-encoded and junk-prone; nbi_to_decimal and
# the validity mask are ported verbatim from derive_coastal_distance.ipynb.
def nbi_to_decimal(val):
    degrees = val // 1000000
    minutes = (val % 1000000) // 10000
    seconds = (val % 10000) / 100
    return degrees + minutes / 60 + seconds / 3600

lat_raw = pd.to_numeric(res["LAT_016"], errors="coerce")
lon_raw = pd.to_numeric(res["LONG_017"], errors="coerce")
lat = nbi_to_decimal(lat_raw)
lon = -nbi_to_decimal(lon_raw)                                  # West = negative
valid = lat_raw.gt(0) & lon_raw.gt(0) & lat.between(15, 72) & lon.between(-170, -60)

map_df = pd.DataFrame({
    "STATE_FIPS": res["STATE_FIPS"], "STRUCTURE_NUMBER_008": res["STRUCTURE_NUMBER_008"],
    "lat": lat.round(5), "lon": lon.round(5),
    "prob_fail_5yr": res["prob_fail_5yr"], "prob_fail_10yr": res["prob_fail_10yr"],
    "prob_fail_20yr": res["prob_fail_20yr"],
})[valid]

# Written to file, never printed — >100 MB of JSON at ~1M bridges.
map_df.to_json(OUT_MAP, orient="records")
print(f"saved {OUT_MAP}: {len(map_df):,} bridges with valid coordinates "
      f"({int((~valid).sum()):,} dropped for bad/missing coords)")

saved us_bridge_risk_map.json: 840,518 bridges with valid coordinates (133,387 dropped for bad/missing coords)


In [14]:
# ── Hyperparameter tuning (guarded by RUN_TUNING) ─────────────────────────────
# Scaled down from MA (50 iters x 5 folds): 15 iters x 3 folds on a 100k subsample with
# 50-tree forests, best params refit on the full training set. Still hours.
if RUN_TUNING:
    from sklearn.model_selection import RandomizedSearchCV
    from scipy.stats import randint

    strat = pd.DataFrame({"state": ids_train["STATE_FIPS"].to_numpy(),
                          "event": y_train["event"]})
    n_tune = min(100_000, len(strat))
    pos = (strat.groupby(["state", "event"], group_keys=False)
                .sample(frac=n_tune / len(strat), random_state=42)).index.to_numpy()
    X_tune, y_tune = X_train.iloc[pos], y_train[pos]

    param_dist = {
        "min_samples_split": randint(50, 300),
        "min_samples_leaf":  randint(20, 100),
        "max_features":      ["sqrt", 0.2, 0.3],
        "max_samples":       [0.3, 0.5, 0.7],
    }
    search = RandomizedSearchCV(
        RandomSurvivalForest(n_estimators=50, low_memory=True, n_jobs=-1, random_state=42),
        param_distributions=param_dist, n_iter=15, cv=3, random_state=42, verbose=2)
    search.fit(X_tune, y_tune)
    print("best params:", search.best_params_, "  search C-index:", round(search.best_score_, 3))
    joblib.dump(search, out_path("us_rsf_random_search.pkl"), compress=3)

    tuned = RandomSurvivalForest(**{**RSF_PARAMS, **search.best_params_})
    tuned.fit(X_train, y_train)
    print(f"tuned C-index (test): {tuned.score(X_test, y_test):.3f}   vs baseline {c_index:.3f}")
    joblib.dump(tuned, out_path("us_rsf_model_tuned.pkl"), compress=3)
else:
    print("skipped (RUN_TUNING = False)")

skipped (RUN_TUNING = False)


In [15]:
# ── Output verification ───────────────────────────────────────────────────────
years = pd.read_csv(OUT_YEARS, dtype={k: str for k in KEYS})
probs = pd.read_csv(OUT_PROBS, dtype={k: str for k in KEYS})

pcols = ["prob_fail_5yr", "prob_fail_10yr", "prob_fail_20yr"]
pv = probs[pcols].dropna()
checks = {
    "row counts match the model matrix":
        len(years) == len(X) and len(probs) == len(X),
    "probabilities all within [0, 1]":
        bool(((pv >= 0) & (pv <= 1)).all().all()),
    "prob_fail_5yr <= prob_fail_10yr <= prob_fail_20yr":
        bool((pv["prob_fail_5yr"] <= pv["prob_fail_10yr"]).all()
             and (pv["prob_fail_10yr"] <= pv["prob_fail_20yr"]).all()),
    "every bridge has a risk_category":
        bool(probs["risk_category"].isin(["High Risk", "Low Risk"]).all()),
}
for name, ok in checks.items():
    print(f"  {'PASS' if ok else 'FAIL'}  {name}")

high = probs["risk_category"] == "High Risk"
print(f"\nHigh-Risk share: {high.mean()*100:.1f}%")
print(f"median years_remaining (High Risk only): "
      f"{years.loc[years['risk_category'] == 'High Risk', 'years_remaining'].median():.0f}")

# MA sanity anchor: the standalone MA model (~20k bridges) is the validated benchmark.
ma_row = state_cidx[state_cidx["STATE_FIPS"] == "25"]
if len(ma_row):
    print(f"\nMA (FIPS 25) inside the national model: n={int(ma_row['n_test'].iloc[0]):,}, "
          f"C-index={ma_row['c_index'].iloc[0]:.3f} (standalone MA benchmark: ../ma_bridges/RSF.ipynb)")

assert all(checks.values()), "one or more output checks FAILED"
print("\nAll output checks passed.")

C:\Users\Joshu\AppData\Local\Temp\ipykernel_6324\2651506442.py:2: DtypeWarning: Columns (0: COUNTY_CODE_003) have mixed types. Specify dtype option on import or set low_memory=False.
  years = pd.read_csv(OUT_YEARS, dtype={k: str for k in KEYS})


  PASS  row counts match the model matrix
  PASS  probabilities all within [0, 1]
  PASS  prob_fail_5yr <= prob_fail_10yr <= prob_fail_20yr
  PASS  every bridge has a risk_category

High-Risk share: 64.1%
median years_remaining (High Risk only): -6

MA (FIPS 25) inside the national model: n=3,836, C-index=0.653 (standalone MA benchmark: ../ma_bridges/RSF.ipynb)

All output checks passed.


C:\Users\Joshu\AppData\Local\Temp\ipykernel_6324\2651506442.py:3: DtypeWarning: Columns (0: COUNTY_CODE_003) have mixed types. Specify dtype option on import or set low_memory=False.
  probs = pd.read_csv(OUT_PROBS, dtype={k: str for k in KEYS})
